# AI Trust Score — Dev Log

## Objetivo e papel no pipeline

`core/trust_score` é o **agregador de confiança** do AthenaGov AI. Sua função é
condensar os sinais produzidos pelos outros módulos de governança — PII Detection,
Policy Engine, Prompt Security e (opcionalmente) Explainability — em um único
número comparável (0 a 100) e um nível de risco categórico (`RiskLevel`), que o
Governance Copilot usa para decidir se uma operação de IA pode prosseguir, precisa
de mitigação, exige revisão humana ou deve ser bloqueada, e que alimenta a seção
de score do RIPD (`RIPDReport.trust_score`) gerado automaticamente.

No pipeline completo do AthenaGov AI, este módulo fica **depois** de PII Detection,
Policy Engine e Prompt Security, e **antes** de Explainability (na direção do dado)
e do RIPD Engine — embora, como veremos, a arquitetura de injeção de dependência
permita que a ordem exata de chamadas seja decidida por quem orquestra (o
Governance Copilot), não por este módulo.

Função pública exportada: `core.trust_score.compute_trust_score()`.

## Fórmula de composição

O score começa em **100.0** (`BASE_SCORE`) e sofre penalidades aditivas de até
quatro fontes independentes. O resultado é sempre limitado (clamp) ao intervalo
`[0, 100]`.

**1. PII (`PIIDetectionResult`).** Cada achado (`PIIFinding`) categorizado como
`DataCategory.SENSITIVE` (Art. 5º, II da LGPD — dado de saúde, biométrico, origem
racial, convicção religiosa etc.) custa **15 pontos**; cada achado `PERSONAL`
(dado comum, Art. 5º, I) custa **4 pontos**. Achados `ANONYMIZED`/`NOT_PERSONAL`
não penalizam. Cada subtotal tem um teto (45 e 20 pontos respectivamente) para
que um documento com dezenas de e-mails, por exemplo, não domine o score de
forma desproporcional a um único vazamento de dado biométrico.

**2. Políticas (`list[PolicyDecision]`).** Este é o fator mais importante porque
reflete uma decisão regulatória já tomada, não apenas um sinal bruto:
- `DENY`: qualquer decisão negada aciona um **piso de 5.0 pontos** — um veto. Não
  importa quão bem as outras dimensões estejam, o score final nunca ultrapassa
  esse piso. Isso é proposital: uma política negada é, por definição, uma
  reprovação da operação, e o score deve refletir isso sem ambiguidade.
- `REQUIRES_HUMAN_REVIEW`: **20 pontos** por ocorrência — incerteza que exige
  supervisão humana é tratada como risco médio-alto.
- `ALLOW_WITH_MITIGATION`: **5 pontos por item** em `decision.mitigations` — a
  penalidade é proporcional ao volume de remediação ainda não comprovadamente
  aplicada (o módulo não tem como saber se as mitigações já foram executadas,
  então penaliza pela quantidade pendente listada).
- `ALLOW`: sem penalidade.

**3. Segurança de prompt (`PromptSecurityResult | None`, opcional).** Quando
informado, soma-se uma penalidade contínua `(1 - score) * 30` mais uma penalidade
fixa de **15 pontos** caso `is_safe is False`. A penalidade fixa existe porque o
próprio módulo de prompt security pode classificar algo como inseguro mesmo com
um score numérico moderado (ex.: um único padrão de jailbreak de alta confiança
detectado, mas sem dominar o score contínuo) — essa classificação binária é um
sinal independente que merece peso próprio.

**4. Explicabilidade (`ExplainabilityResult | None`, opcional).** Não participa
da fórmula numérica. É apenas repassada como veio no campo `explanation` do
resultado — este módulo **nunca gera sua própria narrativa**.

**Mapeamento score -> `RiskLevel`:**

| Score | RiskLevel |
|---|---|
| >= 80 | LOW |
| 50 – 79 | MEDIUM |
| 20 – 49 | HIGH |
| < 20 | CRITICAL |

O piso de `DENY` (5.0) é, por construção, sempre menor que 20 — então uma única
política `DENY` já garante `RiskLevel.CRITICAL` através do mapeamento normal,
sem precisar de nenhum caso especial na atribuição do nível de risco.

O campo `components: dict[str, float]` do `TrustScoreResult` expõe a contribuição
de cada fator (`base_score`, `pii_penalty`, `policy_human_review_penalty`,
`policy_mitigation_penalty`, `policy_deny_penalty`, `prompt_security_penalty`,
`final_score`) — pronto para ser passado como `factors` a `core.explainability`
quando o Governance Copilot compuser a narrativa completa de explicação.

### Por que injeção de dependência em vez de importar os outros módulos

`core/trust_score` está sendo desenvolvido **em paralelo** com `core/pii_detection`,
`core/policy_engine`, `core/prompt_security` e `core/explainability`, cada um por
um agente/desenvolvedor diferente. Se este módulo importasse diretamente qualquer
um deles (`from core.pii_detection import detect`, por exemplo), o código
quebraria sempre que o módulo importado ainda não existisse, estivesse em estado
intermediário, ou tivesse sua API alterada durante o desenvolvimento simultâneo —
uma race condition de integração.

Em vez disso, `compute_trust_score()` depende apenas dos **contratos** definidos
em `shared/schemas.py` (`PIIDetectionResult`, `PolicyDecision`,
`PromptSecurityResult`, `ExplainabilityResult`) — tipos Pydantic estáveis, já
acordados entre todos os módulos antes do início do desenvolvimento paralelo.
Qualquer módulo que produza objetos desses tipos pode alimentar o trust score,
independentemente de como foi implementado por dentro. A integração real —
chamar `pii_detection`, depois `policy_engine`, depois `prompt_security`, e
finalmente `explainability.explain()` com os `components` do trust score
preliminar — é responsabilidade do **Governance Copilot**, o orquestrador que só
é construído na Onda 2, depois que todos os módulos V1 estão prontos e testados
isoladamente. Esse padrão é o mesmo já usado por `core.policy_engine`, que também
consome apenas tipos de `shared.schemas` como entrada/saída.

## Setup

In [1]:
import sys
from pathlib import Path

# Notebook roda a partir de notebooks/ — garante que a raiz do repo (onde ficam
# os pacotes `core` e `shared`) esteja no sys.path.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from shared.schemas import (
    DataCategory,
    ExplainabilityResult,
    PIIDetectionResult,
    PIIFinding,
    PolicyDecision,
    PolicyDecisionStatus,
    PromptSecurityFinding,
    PromptSecurityResult,
    RiskLevel,
)
from core.trust_score.scorer import compute_trust_score

print("Imports OK — REPO_ROOT =", REPO_ROOT)

Imports OK — REPO_ROOT = G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)


## Cenário 1 — bom (sem PII, política ALLOW, prompt seguro)

Nenhum dado pessoal detectado, única decisão de política é `ALLOW`, prompt
classificado como totalmente seguro. Espera-se score máximo e `RiskLevel.LOW`.

In [2]:
pii_result_1 = PIIDetectionResult(
    findings=[],
    has_sensitive_data=False,
    summary="Nenhum dado pessoal ou sensível encontrado no prompt/contexto analisado.",
)

policy_decisions_1 = [
    PolicyDecision(
        policy_id="POL-009",
        status=PolicyDecisionStatus.ALLOW,
        rationale="Dado pessoal comum, base legal determinada, finalidade especificada.",
        mitigations=[],
        risk_level=RiskLevel.LOW,
    )
]

prompt_security_1 = PromptSecurityResult(findings=[], is_safe=True, score=1.0)

result_1 = compute_trust_score(
    pii_result=pii_result_1,
    policy_decisions=policy_decisions_1,
    prompt_security=prompt_security_1,
)

print("=== Cenário 1: bom (sem PII, política ALLOW, prompt seguro) ===")
print(f"score       = {result_1.score}")
print(f"risk_level  = {result_1.risk_level.value}")
print(f"components  = {result_1.components}")
print(f"explanation = {result_1.explanation}")

=== Cenário 1: bom (sem PII, política ALLOW, prompt seguro) ===
score       = 100.0
risk_level  = low
components  = {'base_score': 100.0, 'pii_penalty': -0.0, 'policy_human_review_penalty': -0.0, 'policy_mitigation_penalty': -0.0, 'policy_deny_penalty': -0.0, 'prompt_security_penalty': -0.0, 'final_score': 100.0}
explanation = None


## Cenário 2 — DENY (dado biométrico + decisão automatizada sem revisão humana)

Achado de PII sensível (biométrico) e uma decisão de política `DENY`. Mesmo com
prompt seguro, a política negada deve dominar o resultado: score no piso e
`RiskLevel.CRITICAL`.

In [3]:
pii_result_2 = PIIDetectionResult(
    findings=[
        PIIFinding(
            entity_type="biometric_data",
            text_span="[dado biométrico detectado]",
            start=42,
            end=70,
            category=DataCategory.SENSITIVE,
            confidence=0.97,
        )
    ],
    has_sensitive_data=True,
    summary="1 achado de dado biométrico (sensível, Art. 5º II LGPD).",
)

policy_decisions_2 = [
    PolicyDecision(
        policy_id="POL-002",
        status=PolicyDecisionStatus.DENY,
        rationale="Decisão automatizada com dado biométrico e sem revisão humana configurada.",
        mitigations=[],
        risk_level=RiskLevel.CRITICAL,
    )
]

prompt_security_2 = PromptSecurityResult(findings=[], is_safe=True, score=0.9)

result_2 = compute_trust_score(
    pii_result=pii_result_2,
    policy_decisions=policy_decisions_2,
    prompt_security=prompt_security_2,
)

print("=== Cenário 2: DENY (dado biométrico + decisão automatizada sem revisão humana) ===")
print(f"score       = {result_2.score}")
print(f"risk_level  = {result_2.risk_level.value}")
print(f"components  = {result_2.components}")
print(f"explanation = {result_2.explanation}")

=== Cenário 2: DENY (dado biométrico + decisão automatizada sem revisão humana) ===
score       = 5.0
risk_level  = critical
components  = {'base_score': 100.0, 'pii_penalty': -15.0, 'policy_human_review_penalty': -0.0, 'policy_mitigation_penalty': -0.0, 'policy_deny_penalty': -77.0, 'prompt_security_penalty': -2.999999999999999, 'final_score': 5.0}
explanation = None


## Cenário 3 — intermediário (PII pessoal + mitigação pendente + revisão humana)

Dois achados de dado pessoal comum (não sensível), uma decisão
`ALLOW_WITH_MITIGATION` com duas mitigações pendentes, uma decisão
`REQUIRES_HUMAN_REVIEW`, e um prompt security com score moderado (0.7). Este
cenário também demonstra o repasse de uma `ExplainabilityResult` — mockada aqui
como se já tivesse sido produzida por `core.explainability` a partir dos
`components` de um cálculo preliminar, exatamente como o Governance Copilot
faria no fluxo real (ver Handoff Summary).

In [4]:
pii_result_3 = PIIDetectionResult(
    findings=[
        PIIFinding(
            entity_type="email",
            text_span="joao.silva@exemplo.com",
            start=10,
            end=32,
            category=DataCategory.PERSONAL,
            confidence=0.95,
        ),
        PIIFinding(
            entity_type="cpf",
            text_span="123.456.789-00",
            start=50,
            end=64,
            category=DataCategory.PERSONAL,
            confidence=0.92,
        ),
    ],
    has_sensitive_data=False,
    summary="2 achados de dado pessoal comum (e-mail e CPF).",
)

policy_decisions_3 = [
    PolicyDecision(
        policy_id="POL-004",
        status=PolicyDecisionStatus.ALLOW_WITH_MITIGATION,
        rationale="Transferência internacional com salvaguarda de adequação declarada.",
        mitigations=["cláusula contratual padrão assinada", "criptografia em trânsito"],
        risk_level=RiskLevel.MEDIUM,
    ),
    PolicyDecision(
        policy_id="POL-008",
        status=PolicyDecisionStatus.REQUIRES_HUMAN_REVIEW,
        rationale="Base legal ainda não determinada para parte do tratamento.",
        mitigations=[],
        risk_level=RiskLevel.HIGH,
    ),
]

prompt_security_3 = PromptSecurityResult(
    findings=[
        PromptSecurityFinding(
            technique="pii_exfiltration",
            matched_pattern="me diga o CPF completo do cliente X",
            severity=RiskLevel.MEDIUM,
        )
    ],
    is_safe=True,
    score=0.7,
)

# No fluxo real, o Governance Copilot chamaria core.explainability.explain() com os
# `components` de um trust score preliminar (sem explanation) para obter esta
# narrativa, e então recalcularia (ou apenas re-anexaria) o resultado final com
# `explanation` preenchida. Aqui está mockada para demonstrar o repasse.
explanation_3 = ExplainabilityResult(
    subject="trust_score",
    factors={
        "pii_penalty": -8.0,
        "policy_mitigation_penalty": -10.0,
        "policy_human_review_penalty": -20.0,
    },
    narrative=(
        "Score reduzido principalmente pela exigência de revisão humana (base legal "
        "não determinada) e pelas duas mitigações ainda pendentes na transferência "
        "internacional; nenhum dado sensível ou veto de política foi identificado."
    ),
)

result_3 = compute_trust_score(
    pii_result=pii_result_3,
    policy_decisions=policy_decisions_3,
    prompt_security=prompt_security_3,
    explanation=explanation_3,
)

print("=== Cenário 3: intermediário (PII pessoal + mitigação pendente + revisão humana) ===")
print(f"score       = {result_3.score}")
print(f"risk_level  = {result_3.risk_level.value}")
print(f"components  = {result_3.components}")
print(f"explanation.narrative = {result_3.explanation.narrative}")

=== Cenário 3: intermediário (PII pessoal + mitigação pendente + revisão humana) ===
score       = 53.0
risk_level  = medium
components  = {'base_score': 100.0, 'pii_penalty': -8.0, 'policy_human_review_penalty': -20.0, 'policy_mitigation_penalty': -10.0, 'policy_deny_penalty': -0.0, 'prompt_security_penalty': -9.000000000000002, 'final_score': 53.0}
explanation.narrative = Score reduzido principalmente pela exigência de revisão humana (base legal não determinada) e pelas duas mitigações ainda pendentes na transferência internacional; nenhum dado sensível ou veto de política foi identificado.


## Suíte de testes

Executa `pytest` sobre `core/trust_score/tests` via `subprocess`, usando o mesmo
interpretador do venv do projeto, e mostra o resultado real.

In [5]:
import subprocess
import sys as _sys

proc = subprocess.run(
    [_sys.executable, "-m", "pytest", "core/trust_score/tests", "-v"],
    cwd=str(REPO_ROOT),
    capture_output=True,
    text=True,
)
print(proc.stdout)
print(proc.stderr)
print("return code:", proc.returncode)

============================= test session starts =============================
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: G:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 20 items

core/trust_score/tests/test_scorer.py::test_optimal_case_yields_high_score_and_low_risk PASSED [  5%]
core/trust_score/tests/test_scorer.py::test_optimal_case_with_empty_policy_decisions_is_also_high_score PASSED [ 10%]
core/trust_score/tests/test_scorer.py::test_deny_forces_score_to_floor_and_critical_risk PASSED [ 15%]
core/trust_score/tests/test_scorer.py::test_deny_vetoes_even_when_combined_with_good_signals_elsewhere PASSED [ 20%]
core/trust_score/tests/test_scorer.py::test_requires_human_review_reduces_score_but_does_not_zero_it PASSED [ 25%]
core/

## Handoff Summary

### Capacidades entregues

- `compute_trust_score()` — agregador de AI Trust Score 100% via injeção de
  dependência (não importa `pii_detection`, `policy_engine`, `prompt_security`
  nem `explainability`).
- Fórmula aditiva documentada e testada: penalidade de PII sensível/pessoal com
  tetos, veto por `DENY` (piso de 5.0), penalidade fixa por
  `REQUIRES_HUMAN_REVIEW`, penalidade proporcional por mitigação pendente em
  `ALLOW_WITH_MITIGATION`, penalidade contínua + flag de segurança de prompt.
- Mapeamento score -> `RiskLevel` com limiares documentados e testados nas
  fronteiras (80/50/20).
- `components: dict[str, float]` pronto para virar `factors` de um
  `ExplainabilityResult`.
- `explanation` sempre repassado como recebido, nunca gerado por este módulo.
- Suíte pytest com **20/20 testes passando** cobrindo caso ótimo, veto `DENY`,
  `REQUIRES_HUMAN_REVIEW`, `ALLOW_WITH_MITIGATION` proporcional, PII sensível vs.
  pessoal comum (com tetos), prompt inseguro (contínuo + flag `is_safe`),
  `explanation` ausente vs. repassado, limites do mapeamento de `RiskLevel` e
  contrato de `components` (ver seção de testes acima, saída real do pytest).

### Assinatura pública exata

```python
def compute_trust_score(
    pii_result: PIIDetectionResult,
    policy_decisions: list[PolicyDecision],
    prompt_security: PromptSecurityResult | None = None,
    explanation: ExplainabilityResult | None = None,
) -> TrustScoreResult:
    ...
```

Importável tanto como `from core.trust_score import compute_trust_score` quanto
`from core.trust_score.scorer import compute_trust_score`.

### Como o Governance Copilot deve chamar (fluxo completo — pseudo-código)

```python
from core.pii_detection import detect            # nome ilustrativo — ver API real do módulo
from core.policy_engine import evaluate
from core.prompt_security import scan            # nome ilustrativo — ver API real do módulo
from core.explainability import explain           # nome ilustrativo — ver API real do módulo
from core.trust_score import compute_trust_score

# 1. PII Detection
pii_result = detect(text=prompt_text)

# 2. Policy Engine
policy_decisions = evaluate(
    data_categories=data_categories_from(pii_result),
    legal_basis=declared_legal_basis,
    context=operational_context,
)

# 3. Prompt Security
prompt_security_result = scan(prompt_text)

# 4a. Trust Score preliminar (sem explanation) — dá os `components` que
#     alimentam a explicação.
preliminary = compute_trust_score(
    pii_result=pii_result,
    policy_decisions=policy_decisions,
    prompt_security=prompt_security_result,
)

# 4b. Explainability — gera a narrativa a partir dos components do trust score.
explanation = explain(subject="trust_score", factors=preliminary.components)

# 4c. Trust Score final — mesmos inputs, agora com explanation preenchida.
final_trust_score = compute_trust_score(
    pii_result=pii_result,
    policy_decisions=policy_decisions,
    prompt_security=prompt_security_result,
    explanation=explanation,
)

# 5. RIPD Engine consome final_trust_score.trust_score (via RIPDReport)
```

O recálculo em 4c é barato (função pura, sem I/O) e evita ter que remontar o
objeto `TrustScoreResult` manualmente — mas o Governance Copilot também pode
optar por copiar `preliminary.model_copy(update={"explanation": explanation})`
se preferir evitar a segunda chamada.

### Limitações

- A fórmula é determinística e baseada em regras fixas (pesos e tetos definidos
  no código), não em um modelo estatístico calibrado com dados reais de
  incidentes — os pesos são justificáveis por design, mas não validados
  empiricamente.
- Não há normalização por volume de texto/contexto: um prompt de 20 palavras
  com um dado sensível recebe a mesma penalidade fixa que um documento de 5.000
  palavras com um único dado sensível.
- `ALLOW_WITH_MITIGATION` penaliza pela contagem de mitigações *listadas*, não
  por sua *aplicação verificada* — o módulo não tem como confirmar se uma
  mitigação já foi implementada.
- Múltiplas políticas `DENY` simultâneas não são distinguidas de uma única
  (o piso é o mesmo) — não há gradação de "quão negado".

### O que fica para V2

- Calibração dos pesos/tetos da fórmula com dados reais (ex.: histórico de
  incidentes ou feedback de auditores) em vez de valores fixados por design.
- Ponderação por confiança (`PIIFinding.confidence`, `PromptSecurityFinding`
  severity) em vez de contagem simples de achados.
- Séries temporais de trust score por agente/aplicação (AI Observability, já
  listado no ROADMAP V2) para detectar degradação de confiança ao longo do
  tempo, não apenas um snapshot por chamada.
- Integração com Fairness Audit (V2) como quinto fator de composição.